# 🗂️ Notebook 2: Key-Value Store — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

Keep it minimal and byte-oriented. The store doesn't care what's inside the value.

```
key     : bytes (≤ 1 KB)     — short, unique identifier
value   : bytes (≤ 1 MB)     — opaque blob (JSON, protobuf, image, whatever)
version : monotonic counter  — used to resolve conflicts
```

**Why bytes?** The store is schema-less. The *application* encodes/decodes (JSON,
Avro, protobuf). The store only has to copy, index, and return bytes.


In [1]:
from pydantic import BaseModel, Field

class KVRecord(BaseModel):
    key: str = Field(..., max_length=1024)
    value: str  # in practice: bytes; we use str for easy printing
    version: int = Field(..., ge=0)

r = KVRecord(key="user:42", value="alice", version=7)
print(r.model_dump_json(indent=2))

{
  "key": "user:42",
  "value": "alice",
  "version": 7
}


## HTTP API

```http
PUT    /kv/{key}   body: value         headers: X-Consistency: one|quorum|all
GET    /kv/{key}   → { value, version }
DELETE /kv/{key}
```

Internally, nodes gossip cluster membership and run **anti-entropy** (Merkle tree
diffs — Notebook 3) to reconcile replicas that drifted apart.


In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class PutRequest(BaseModel):
    key: str = Field(..., max_length=1024)
    value: str
    consistency: Literal["one", "quorum", "all"] = "quorum"

class GetResponse(BaseModel):
    key: str
    value: str | None
    version: int | None
    replicas_answered: int

req = PutRequest(key="user:42", value="alice")
resp = GetResponse(key="user:42", value="alice", version=7, replicas_answered=2)
print("PUT  request :", req.model_dump_json())
print("GET  response:", resp.model_dump_json())

PUT  request : {"key":"user:42","value":"alice","consistency":"quorum"}
GET  response: {"key":"user:42","value":"alice","version":7,"replicas_answered":2}


## 😱 Bad practice: storing raw values with no versioning

If the store keeps just `key → value` and nothing else, two concurrent writes
race and the loser is silently overwritten. Worse: we can't even *tell* a conflict
happened.


In [3]:
# Single replica, no versioning — "last write wins" by wall clock order
store: dict[str, str] = {}

# Alice and Bob both read cart, add one item each, write back.
store["cart:42"] = "[apple]"          # initial

alice = store["cart:42"] + ",banana"  # Alice reads, appends
bob   = store["cart:42"] + ",cherry"  # Bob reads, appends
# Both write back, Bob arrives last:
store["cart:42"] = alice
store["cart:42"] = bob

print("final cart:", store["cart:42"])
print("➡ Alice's banana is LOST, silently. No warning, no log.")

final cart: [apple],cherry
➡ Alice's banana is LOST, silently. No warning, no log.


## ✅ Better: version numbers (last-write-wins, but *detectable*)

Every write bumps a version. A replica keeps whichever version number is higher.
This is simple and cheap, and it's what DynamoDB uses by default. It still loses
data on concurrent updates — the app must know this — but at least the store can
warn you when versions are stale.


In [4]:
class LWWStore:
    def __init__(self):
        self.data: dict[str, tuple[str, int]] = {}

    def put(self, key, value, version):
        cur = self.data.get(key)
        if cur is None or version > cur[1]:
            self.data[key] = (value, version)
            return "written"
        return "rejected-stale"

    def get(self, key):
        return self.data.get(key)

s = LWWStore()
print(s.put("cart:42", "[apple]",          version=1))
print(s.put("cart:42", "[apple,banana]",   version=2))  # Alice
print(s.put("cart:42", "[apple,cherry]",   version=2))  # Bob — SAME version = conflict!
print("stored :", s.get("cart:42"))
print("➡ Bob's write is rejected because versions tied. App can retry with v3.")

written
written
rejected-stale
stored : ('[apple,banana]', 2)
➡ Bob's write is rejected because versions tied. App can retry with v3.


## 🌟 Best: vector clocks (preserve both sides of a conflict)

A vector clock is a dict `{node_id: counter}`. Every node increments its own
counter on write. On read, we compare clocks:

- `A` dominates `B`  →  A is strictly newer, keep A.
- Neither dominates  →  **concurrent writes** — return *both* as *siblings*
  and let the app merge (e.g., union the shopping carts).

This is how Dynamo and Riak avoid silent data loss.


In [5]:
from dataclasses import dataclass, field

@dataclass
class Versioned:
    value: str
    clock: dict[str, int] = field(default_factory=dict)

def dominates(a: dict, b: dict) -> bool:
    """True if clock a is >= b on every node AND strictly greater somewhere."""
    all_ge = all(a.get(k, 0) >= v for k, v in b.items())
    any_gt = any(a.get(k, 0) >  b.get(k, 0) for k in set(a) | set(b))
    return all_ge and any_gt

def merge_siblings(values: list[Versioned]) -> list[Versioned]:
    """Keep only values whose clock is NOT dominated by another."""
    keep = []
    for v in values:
        if not any(dominates(w.clock, v.clock) for w in values if w is not v):
            keep.append(v)
    return keep

# Alice (node A) and Bob (node B) each write concurrently based on v1={A:1}
v1      = Versioned("[apple]",          clock={"A": 1})
alice   = Versioned("[apple,banana]",   clock={"A": 2})            # A advanced
bob     = Versioned("[apple,cherry]",   clock={"A": 1, "B": 1})    # B advanced

siblings = merge_siblings([alice, bob])
print(f"{len(siblings)} siblings returned to client:")
for s in siblings:
    print(" ", s)
print("➡ App merges → [apple, banana, cherry]. Nothing lost!")

2 siblings returned to client:
  Versioned(value='[apple,banana]', clock={'A': 2})
  Versioned(value='[apple,cherry]', clock={'A': 1, 'B': 1})
➡ App merges → [apple, banana, cherry]. Nothing lost!


## 🗄️ Storage layer — a glimpse of LSM-trees

Inside each node, how do we actually persist bytes on disk? Two classic options:

- **B-tree** (MySQL, Postgres) — great for reads, slower writes (random I/O).
- **LSM-tree** (Cassandra, RocksDB, LevelDB) — great for writes, reads need merging.

An LSM-tree writes:
1. **WAL** (write-ahead log): append the op to a file, fsync → durable.
2. **Memtable**: keep recent writes in a sorted in-memory structure.
3. When the memtable fills, **flush** it as an immutable sorted file (SSTable).
4. A background **compaction** merges small SSTables into larger ones.

KV stores are write-heavy, so LSM usually wins. Here's a toy version:


In [6]:
import os, tempfile, json

class ToyLSM:
    def __init__(self, path):
        self.wal_path = path
        self.memtable: dict[str, str] = {}
        self.sstables: list[dict[str, str]] = []
        # Replay WAL (simulates crash recovery)
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    op = json.loads(line)
                    self.memtable[op["k"]] = op["v"]

    def put(self, k, v):
        with open(self.wal_path, "a") as f:   # 1. append to WAL (durable)
            f.write(json.dumps({"k": k, "v": v}) + "\n")
        self.memtable[k] = v                  # 2. update memtable

    def get(self, k):
        if k in self.memtable:                # newest wins
            return self.memtable[k]
        for sst in reversed(self.sstables):   # search SSTables (newest first)
            if k in sst:
                return sst[k]
        return None

    def flush(self):
        if self.memtable:
            self.sstables.append(self.memtable)
            self.memtable = {}
            open(self.wal_path, "w").close()  # truncate WAL

with tempfile.TemporaryDirectory() as d:
    db = ToyLSM(os.path.join(d, "wal.log"))
    db.put("a", "1"); db.put("b", "2"); db.flush()
    db.put("a", "99")                         # updates in memtable
    print("get a =", db.get("a"))             # 99 (memtable beats SSTable)
    print("get b =", db.get("b"))             # 2  (flushed SSTable)

get a = 99
get b = 2


## Recap

- Schema-less: key/value are opaque bytes; store doesn't care.
- No version → **silent data loss** on concurrent writes.
- LWW versions → detectable conflicts but still lossy.
- Vector clocks → preserve siblings, let the app merge.
- Real nodes store data in **LSM-trees** (WAL + memtable + SSTables) for fast writes.

Next notebook: deep dives into consistent hashing, quorums, and Merkle-tree anti-entropy.
